In [1]:
!pip install yagmail
!pip install dnspython



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 4.1 MB/s eta 0:00:00


In [8]:
# 1. MONTAR GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# 2. DEFINIR CARPETA DE TRABAJO
import os
carpeta_drive = '/content/drive/MyDrive/TinkuDevMailing'  # Puedes cambiar el nombre
os.makedirs(carpeta_drive, exist_ok=True)  # Crea si no existe

In [11]:
# 3. SUBIR ARCHIVOS Y GUARDARLOS EN ESA CARPETA
from google.colab import files

print("🔼 Sube archivos como el PDF, PNG o .db")
uploaded = files.upload()

for nombre_archivo in uploaded.keys():
    ruta_completa = os.path.join(carpeta_drive, nombre_archivo)
    with open(ruta_completa, 'wb') as f:
        f.write(uploaded[nombre_archivo])
    print(f"✅ Guardado en: {ruta_completa}")


🔼 Sube archivos como el PDF, PNG o .db


Saving Firma Isaac Haro.png to Firma Isaac Haro.png
✅ Guardado en: /content/drive/MyDrive/TinkuDevMailing/Firma Isaac Haro.png


In [12]:
# 4. MOSTRAR ARCHIVOS GUARDADOS
print("\n📁 Archivos guardados en TinkuDevMailing:")
print(os.listdir(carpeta_drive))


📁 Archivos guardados en TinkuDevMailing:
['flyer tinkudev.pdf', '.ipynb_checkpoints', 'colegios.db', 'Firma Isaac Haro.png']


In [13]:
# 5. CONECTAR A LA BASE DE DATOS (desde Drive)
import sqlite3
import pandas as pd

ruta_db = os.path.join(carpeta_drive, "colegios.db")  # Asegúrate de haberlo subido
conn = sqlite3.connect(ruta_db)

df = pd.read_sql_query("SELECT * FROM colegios ORDER BY enviado, nombre", conn)

# Mostrar como tabla bonita
from IPython.display import display
print("📊 Estado completo de la base de datos:")
display(df)

pendientes = df[df["enviado"] == "no"]
print(f"\n🔔 Correos pendientes por enviar: {len(pendientes)}")

conn.close()

📊 Estado completo de la base de datos:


,id,nombre,correo,enviado
0,572,ASECCBI,coordinacion@aseccbi.edu.ec,no
1,607,Agencias Postales,canaldegestion@serviciopostal.gob.ec,no
2,434,CEEAL,admisiones@ceeal.edu.ec,no
3,435,CEEAL,colecturia@soyamericalatina.edu.ec,no
4,436,CEEAL,info@funlif.org,no
...,...,...,...,...
729,56,Colegio Torremar,rector@torremar.edu.ec,sí
730,57,Colegio Torremar,vicerrector@torremar.edu.ec,sí
731,58,Colegio Torremar,biblioteca@torremar.edu.ec,sí
732,59,Colegio Torremar,direccion@torremar.edu.ec,sí



🔔 Correos pendientes por enviar: 690


In [16]:
# 1. Ruta completa a la base de datos dentro de Drive
ruta_db = os.path.join(carpeta_drive, "colegios.db")

# 2. Crear/conectar la base de datos en Drive
import sqlite3
import re
import dns.resolver

conn = sqlite3.connect(ruta_db)
cursor = conn.cursor()

# 3. Crear tabla si no existe
cursor.execute("""
CREATE TABLE IF NOT EXISTS colegios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT,
    correo TEXT UNIQUE,
    enviado TEXT DEFAULT 'no'
)
""")
conn.commit()

# 4. Funciones de validación
def correo_valido(correo):
    patron = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
    return re.match(patron, correo) is not None

def dominio_valido(correo):
    try:
        dominio = correo.split('@')[1]
        registros_mx = dns.resolver.resolve(dominio, 'MX')
        return len(registros_mx) > 0
    except:
        return False

# 5. Lista de colegios (usa la que ya tienes)
colegios = [
  {"nombre": "Equipo Educativo", "correos": "carlospaspuel@educacionsiglo21.onmicrosoft.com"},
  {"nombre": "Equipo Educativo", "correos": "eduardo.cardenas@educacion.gob.ec"},
]

# 6. Insertar correos válidos
for colegio in colegios:
    nombre = colegio["nombre"]
    correos = [c.strip() for c in colegio["correos"].split(",") if c.strip()]

    for correo in correos:
        if correo_valido(correo) and dominio_valido(correo):
            try:
                cursor.execute("INSERT INTO colegios (nombre, correo) VALUES (?, ?)", (nombre, correo))
            except sqlite3.IntegrityError:
                pass  # Ya existe

conn.commit()
conn.close()

print(f"✅ Base de datos creada y guardada en: {ruta_db}")

✅ Base de datos creada y guardada en: /content/drive/MyDrive/TinkuDevMailing/colegios.db


In [17]:
# 5. CONECTAR A LA BASE DE DATOS (desde Drive)
import sqlite3
import pandas as pd

ruta_db = os.path.join(carpeta_drive, "colegios.db")  # Asegúrate de haberlo subido
conn = sqlite3.connect(ruta_db)

df = pd.read_sql_query("SELECT * FROM colegios ORDER BY enviado, nombre", conn)

# Mostrar como tabla bonita
from IPython.display import display
print("📊 Estado completo de la base de datos:")
display(df)

pendientes = df[df["enviado"] == "no"]
print(f"\n🔔 Correos pendientes por enviar: {len(pendientes)}")

conn.close()

📊 Estado completo de la base de datos:


,id,nombre,correo,enviado
0,572,ASECCBI,coordinacion@aseccbi.edu.ec,no
1,607,Agencias Postales,canaldegestion@serviciopostal.gob.ec,no
2,434,CEEAL,admisiones@ceeal.edu.ec,no
3,435,CEEAL,colecturia@soyamericalatina.edu.ec,no
4,436,CEEAL,info@funlif.org,no
...,...,...,...,...
729,56,Colegio Torremar,rector@torremar.edu.ec,sí
730,57,Colegio Torremar,vicerrector@torremar.edu.ec,sí
731,58,Colegio Torremar,biblioteca@torremar.edu.ec,sí
732,59,Colegio Torremar,direccion@torremar.edu.ec,sí



🔔 Correos pendientes por enviar: 690


In [ ]:
# 1. MONTAR GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

# 2. IMPORTAR LIBRERÍAS NECESARIAS
import sqlite3
import re
import dns.resolver
import yagmail
import time
from datetime import datetime
import os

# 3. DEFINIR CARPETA EN DRIVE
carpeta_drive = '/content/drive/MyDrive/TinkuDevMailing'
ruta_db = os.path.join(carpeta_drive, 'colegios.db')
pdf_adjunto = os.path.join(carpeta_drive, 'flyer tinkudev.pdf')
firma_imagen = os.path.join(carpeta_drive, 'Firma Isaac Haro.png')

# 4. CONECTAR A LA BASE DE DATOS
conn = sqlite3.connect(ruta_db)
cursor = conn.cursor()

# 5. CONFIGURACIÓN DE GMAIL
usuario = "tinkudevec@gmail.com"
clave = "oowd tpgh reur kikq"  # Recuerda: esta debe ser una contraseña de aplicación generada desde Gmail

# 6. HTML DEL CORREO (usa {nombre_colegio} como marcador)
mensaje_html = """
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8" />
  <title>TinkuDev - Propuesta</title>
</head>
<body style="margin: 0; padding: 0; background-color: #e5e5e5;">
  <table width="100%" cellpadding="0" cellspacing="0" bgcolor="#e5e5e5" style="padding: 20px 0; font-family: 'Segoe UI', sans-serif;">
    <tr>
      <td align="center">
        <table width="700" cellpadding="0" cellspacing="0" style="background-color: #ffffff; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 20px rgba(0, 0, 0, 0.15);">
          <tr>
            <td style="background-color: #2f9e44; padding: 20px; text-align: center;">
              <img src="https://drive.google.com/uc?export=view&id=12yrQZULihREfZVOSOGey-F3R0VWR1CgJ" alt="Logo TinkuDev" width="80" style="vertical-align: middle; margin-right: 10px;" />
              <span style="color: white; font-size: 24px; font-weight: bold; vertical-align: middle;">
                Tinku<span style="color: black;">Dev</span>
              </span>
              <p style="color: #d4f4dd; font-size: 13px; margin: 5px 0 0;">Soluciones tecnológicas educativas</p>
            </td>
          </tr>
          <tr>
            <td style="padding: 30px; color: #333; font-size: 16px; line-height: 1.4;">
              <p style="margin: 0 0 10px;">Hola equipo del <strong>{nombre_colegio}</strong>,</p>
              <p style="margin: 0 0 10px;">
                Mi nombre es Isaac Haro, CEO de <strong>TinkuDev</strong>, empresa ecuatoriana especializada en soluciones tecnológicas educativas.
              </p>
              <p style="margin: 0 0 10px;">
                En la actualidad, los colegios enfrentan el reto de digitalizar procesos, fomentar la producción académica y cumplir con estándares de calidad. Por eso, ofrecemos herramientas de clase mundial para fortalecer su institución:
              </p>
              <ul style="padding-left: 20px; margin: 0 0 10px;">
                <li><strong>DSpace:</strong> Repositorio institucional para trabajos y proyectos.</li>
                <li><strong>Koha:</strong> Sistema moderno de gestión bibliotecaria.</li>
                <li><strong>OJS:</strong> Plataforma editorial para revistas y boletines académicos.</li>
              </ul>
              <p style="margin: 0 0 10px;">
                Estas plataformas son utilizadas por instituciones líderes en el mundo. Mejoran la organización, el prestigio y el acceso a la información.
              </p>
              <p style="margin: 0; font-weight: bold; color: #000;">
                Nos encantaría enviarles una propuesta gratuita y personalizada para su institución.
              </p>
              <div style="text-align: center; margin: 20px 0 0;">
                <a href="mailto:tinkudevec@gmail.com" style="background-color: #2f9e44; color: white; text-decoration: none; padding: 12px 24px; border-radius: 6px; font-size: 16px; font-weight: bold;">📩 Solicitar Propuesta</a>
              </div>
            </td>
          </tr>
          <tr>
            <td style="background-color: #f3f3f3; color: #333; padding: 15px 20px; text-align: center; font-size: 14px;">
              <p style="margin: 0;">
                <strong>Isaac Haro</strong><br>
                CEO & Co-Founder – TinkuDev<br>
                📧 tinkudevec@gmail.com | 📞 +593 98 805 5517
              </p>
              <img src="cid:Firma Isaac Haro.png" alt="Firma" style="margin-top: 10px; max-width: 140px;">
            </td>
          </tr>
        </table>
      </td>
    </tr>
  </table>
</body>
</html>
"""

# 7. CONFIGURAR ENVÍO DE CORREO
yag = yagmail.SMTP(usuario, clave)

# 8. OBTENER LOS PENDIENTES
cursor.execute("SELECT id, nombre, correo FROM colegios WHERE enviado = 'no'")
pendientes = cursor.fetchall()

print(f"📬 Correos por enviar: {len(pendientes)}\n")

# 9. ENVIAR UNO A UNO
for id_, nombre, correo in pendientes:
    try:
        html_personalizado = mensaje_html.replace("{nombre_colegio}", nombre)

        yag.send(
            to=correo,
            subject="Modernice su institución con DSpace, Koha y OJS – sin complicaciones",
            contents=[html_personalizado, yagmail.inline(firma_imagen)],
            attachments=pdf_adjunto
        )

        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Enviado a: {correo}")

        # Marcar como enviado
        cursor.execute("UPDATE colegios SET enviado = 'sí' WHERE id = ?", (id_,))
        conn.commit()

    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ❌ Error al enviar a {correo}: {e}")

    print("⏳ Esperando 5 minutos...\n")
    time.sleep(300)  # 5 minutos = 300 segundos

# 10. CERRAR CONEXIÓN
conn.close()
print("📁 Finalizado.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📬 Correos por enviar: 690

[11:14:45] ✅ Enviado a: info@sscc.edu.ec
⏳ Esperando 5 minutos...



In [18]:
# 5. CONECTAR A LA BASE DE DATOS (desde Drive)
import sqlite3
import pandas as pd

ruta_db = os.path.join(carpeta_drive, "colegios.db")  # Asegúrate de haberlo subido
conn = sqlite3.connect(ruta_db)

df = pd.read_sql_query("SELECT * FROM colegios ORDER BY enviado, nombre", conn)

# Mostrar como tabla bonita
from IPython.display import display
print("📊 Estado completo de la base de datos:")
display(df)

pendientes = df[df["enviado"] == "no"]
print(f"\n🔔 Correos pendientes por enviar: {len(pendientes)}")

conn.close()

📊 Estado completo de la base de datos:


,id,nombre,correo,enviado
0,572,ASECCBI,coordinacion@aseccbi.edu.ec,no
1,607,Agencias Postales,canaldegestion@serviciopostal.gob.ec,no
2,434,CEEAL,admisiones@ceeal.edu.ec,no
3,435,CEEAL,colecturia@soyamericalatina.edu.ec,no
4,436,CEEAL,info@funlif.org,no
...,...,...,...,...
729,56,Colegio Torremar,rector@torremar.edu.ec,sí
730,57,Colegio Torremar,vicerrector@torremar.edu.ec,sí
731,58,Colegio Torremar,biblioteca@torremar.edu.ec,sí
732,59,Colegio Torremar,direccion@torremar.edu.ec,sí



🔔 Correos pendientes por enviar: 690
